# Two-Tower Model — complementary products

**The goal of this notebook is to build a two-tower neural network (TTN) that
finds complementary products** — given an item a user is looking at, retrieve
the items that are bought *alongside* it rather than the items most similar to
it. A phone case complements a phone; another phone does not.

The approach follows **[Suggest, complement, inspire: story of Two Tower
recommendations at Allegro.com](https://arxiv.org/html/2508.03702v1)**
(Osowska-Kurczab, Nazarko, Marzec, Wojciechowska & Kremeňová, RecSys '25),
whose Complementary-TT model is the architecture this work is based on.

## Both towers describe items

This is the part that differs from the classic user/item two-tower setup, and
it shapes every column decision below: **the query tower and the candidate
tower both consume item information.** Neither tower is a user tower.

```
   query ITEM features                    candidate ITEM features
        │                                          │
   ┌────▼────┐                                ┌────▼────┐
   │  QUERY  │  product encoder               │CANDIDATE│  product encoder
   │  TOWER  │  (+ target category)           │  TOWER  │
   └────┬────┘                                └────┬────┘
        │                                          │
   q ∈ ℝ^d  ──────────  score = q · c  ──────────  c ∈ ℝ^d
```

Both towers share the same *architecture* — the paper's "Product Encoder":
each item attribute goes through its own embedding table, the vectors are
concatenated, passed through an MLP and L2-normalised. In the paper the query
tower is the only one modified for the complementary task: the query product
embedding is concatenated with a **target category embedding** drawn from a
one-to-many complementary-category mapping, while the candidate tower stays a
plain product encoder.

That mapping is what `complementary_cats_pairs/` produces —
`data/complementary_categories.pkl`, source category path → target category path,
scored by support and lift. §1 loads it. The co-purchase pairs that supply the
training positives come from the same package's `pairs.ipynb`.

Because both sides are items, per-user history is not a tower input at all and
this notebook does not load it. A user's history still shapes the data — it is
what defines which items count as co-purchased — but that work happens upstream,
in `complementary_cats_pairs/pairs.ipynb`.

## What this notebook covers

It loads the tables the model needs and turns them into training pairs. §1
reads everything; §4 produces `tower_pairs`, one row per directed
(query item, candidate item) example:

| Column | |
| --- | --- |
| `asin_query`, `query_cat_2/3/4` | the item being looked at |
| `asin_target`, `target_cat_2/3/4` | an item bought alongside it, whose category the mapping licenses |

The train/test split is **not** applied here. `co_purchase_pairs.pkl` is
already built from interactions before `date_threshold` (see
`ttn/constants.json`), so the positives in §4 are leak-free by construction.
Model definition and training come next.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# data/ lives at the repo root, one level up from this ttn/ folder
ROOT = Path("..").resolve()
DATA_DIR = ROOT / "data"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Show every column/variable when displaying a dataframe
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 50)
# Turn off scientific notation (e.g. 2.447268e+06 -> 2447268.00)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Load the datasets

Everything this notebook reads, in one place. Nothing below this section opens
a file — every later cell transforms tables that are already in memory.

| Dataset | Grain | What it is |
| --- | --- | --- |
| `Home_and_Kitchen_filtered.csv` | one row per review | the interaction log — who bought what, when |
| `df_features.pkl` | one row per `asin` | extracted item attributes: `cat_*`, `brand`, `title_cleaned`, the per-field columns and their parsed measures |
| `co_purchase_pairs.pkl` | one row per item pair | items the same user bought within 90 days, before the cutoff — the **training positives** |
| `complementary_categories.pkl` | one row per directed category pair | which category buys into which, filtered by support and lift — the **complementary mapping** |
| `meta_Home_and_Kitchen_filtered.csv` | one row per `asin` | the *unfiltered* catalogue; describes 150,826 items `df_features` does not |

`asin` and `reviewerID` are pinned to `str` throughout so ids with leading
zeros (e.g. `0560467893`) survive the read.

Only five of the catalogue's fifteen columns are read. `df_features` already
carries every field the catalogue has — the catalogue's value here is
**coverage**, not extra columns: it describes 28,537 reviewed asins that have
no `df_features` row, every one of them in a `cat_3` with no extraction schema.
Reading all fifteen columns of a 2.1 GB file to use four of them is waste.

All five together peak at about **4.4 GB** of RAM.


In [ ]:
# --- 1. Interactions: one row per review ----------------------------------
df_reviews = pd.read_csv(
    DATA_DIR / "Home_and_Kitchen_filtered.csv",
    dtype={"asin": str, "reviewerID": str},
    low_memory=False,
)

# --- 2. Item features: one row per asin, the extracted attributes ---------
df_features = pd.read_pickle(DATA_DIR / "df_features.pkl")

# --- 3. Co-purchase pairs: the training positives -------------------------
co_pairs = pd.read_pickle(DATA_DIR / "co_purchase_pairs.pkl")

# --- 4. The complementary category mapping --------------------------------
comp_cat = pd.read_pickle(DATA_DIR / "complementary_categories.pkl")

# --- 5. The unfiltered catalogue: coverage for items df_features lacks ----
meta_catalogue = pd.read_csv(
    DATA_DIR / "meta_Home_and_Kitchen_filtered.csv",
    usecols=["asin", "category", "title", "brand", "price"],
    dtype={"asin": str},
    low_memory=False,
)

for name, frame in [
    ("df_reviews", df_reviews), ("df_features", df_features),
    ("co_pairs", co_pairs), ("comp_cat", comp_cat),
    ("meta_catalogue", meta_catalogue),
]:
    print(f"{name:<18} {str(frame.shape):>18}")

print(f"\nunique users: {df_reviews['reviewerID'].nunique():,} | "
      f"unique items: {df_reviews['asin'].nunique():,}")
print(f"items described by df_features : {df_features['asin'].nunique():,}")
print(f"items described by the catalogue: {meta_catalogue['asin'].nunique():,}")
df_reviews.head(5)

## 2. Validate the feature table

Before anything is joined, check that `df_features.pkl` is what the pipeline
promised: the exact expected column set, one row per `asin`, numeric columns
actually numeric, coverage above its floors, unit columns free of new values,
and every `_cleaned` column inside its bound.

A **FAIL on `columns`** is the one to care about most — it means a feature
appeared that nothing describes, or one silently disappeared.

In [3]:
from feature_extraction_workflow.validations import run_all

report = run_all(df_features, DATA_DIR / "master_metadata.json")
display(report if len(report) else "no findings")

1,134,566 rows x 92 columns — 1 failure(s), 0 warning(s)


,check,level,subject,detail
0,columns,FAIL,also_buy,present in the table but not in the contract


## 3. Clean the item category path

`cat_4_clean` is built from `data/category_taxonomy.json` — a reviewed
whitelist of which `(cat_3, cat_4)` pairs are real categories rather than
product bullets that leaked into the path. 921 → 451 distinct values, with
0.1% of items landing in a `<cat_3>_Other` bucket.

This runs before anything else because the complementary mapping's `cat_4`
values are folded the same way. Joining §4 on the raw `cat_4` would match on
spelling rather than meaning and drop roughly one valid pair in six.


In [4]:
import json

TAXONOMY_PATH = DATA_DIR / "category_taxonomy.json"
with open(TAXONOMY_PATH) as f:
    taxonomy = json.load(f)

# cat_3 -> the set of cat_4 values that survived the review
valid_cat_4 = {c3: set(vals) for c2 in taxonomy for c3, vals in taxonomy[c2].items()}
print(f"taxonomy: {len(taxonomy)} cat_2 | {len(valid_cat_4)} cat_3 | "
      f"{sum(len(v) for v in valid_cat_4.values())} valid cat_4 slots")

MISSING = "Missing"
OTHER_SUFFIX = "_Other"

cat_3 = df_features["cat_3"].astype(str)
cat_4 = df_features["cat_4"].fillna(MISSING).astype(str)

# A value is kept only if it is valid *under its own parent* — the same label
# can be real in one branch and junk in another.
valid_pairs = {(c3, v) for c3, vals in valid_cat_4.items() for v in vals}
keep = pd.Series(list(zip(cat_3, cat_4)), index=df_features.index).isin(valid_pairs)

df_features["cat_4_clean"] = np.where(keep, cat_4, cat_3 + OTHER_SUFFIX)

n_before = df_features["cat_4"].nunique(dropna=False)
n_after = df_features["cat_4_clean"].nunique()
n_folded = int((~keep).sum())
print(f"\ndistinct cat_4 : {n_before:,} -> {n_after:,}")
print(f"items folded into '<cat_3>{OTHER_SUFFIX}': {n_folded:,} "
      f"({n_folded / len(df_features) * 100:.2f}% of the catalog)")
df_features[["asin", "cat_2", "cat_3", "cat_4", "cat_4_clean"]].head(5)

taxonomy: 7 cat_2 | 69 cat_3 | 521 valid cat_4 slots

distinct cat_4 : 921 -> 451
items folded into '<cat_3>_Other': 1,099 (0.10% of the catalog)


,asin,cat_2,cat_3,cat_4,cat_4_clean
0,0001487795,Kitchen & Dining,Dining & Entertaining,Dinnerware,Dinnerware
1,0002020300,Home Dcor,Candles & Holders,Candles,Candles
2,0006564224,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,Glassware & Drinkware
3,0009046461,Bath,Bathroom Accessories,None,Missing
4,0234937912,Home Dcor,Home Fragrance,Incense & Incense Holders,Incense & Incense Holders


## 4. Preparing the tower data

The first stage of turning the tables above into training examples. Three
steps, no modelling yet — the query/target *roles* are assigned here, but what
each tower does with them comes later.

1. **Join the categories onto both ends.** `co_purchase_pairs.pkl` is just two
   asins; `df_features` supplies each one's category path. Inner join on both
   sides, so a pair survives only if both items have features. Result: 8
   columns — two asins and three category levels each.
2. **Inner join onto the mapping, in order.** `asinA`'s three categories against
   the mapping's *first* three (`src_*`), `asinB`'s against the *last* three
   (`dst_*`). A row survives only where that exact directed category relation
   exists. Here `asinA` is the source, so it becomes the **query**.
3. **Inner join again, reversed.** `asinA`'s categories against `dst_*` and
   `asinB`'s against `src_*`. Now `asinB` is the source, so `asinB` becomes the
   **query** and `asinA` the target.

Then concatenate. Both tables are relabelled so the source side is always
`asin_query` / `query_cat_*` and the target side always `asin_target` /
`target_cat_*`, which is what makes the concat meaningful — otherwise the
query would sit in a different column in each half.

A pair licensed in both directions appears in both tables. That is not
duplication: `X → Y` and `Y → X` are two different training examples, and
which item is the query differs between them.

`edges`, `support` and `lift` are dropped. They did their job when the mapping
was filtered; the model does not consume them.

**`cat_4_clean`, not `cat_4`.** The mapping's `cat_4` values are folded through
`category_taxonomy.json`, and §2 already computes that fold (identical to the
package's `fold_cat_4`). An inner join on the raw column would match on
spelling rather than meaning and drop roughly one valid pair in six.


In [5]:
# --- Step 1: the pair, plus the category path of each end ------------------
CAT_LEVELS = ["cat_2", "cat_3", "cat_4_clean"]     # the mapping's three levels

item_cats = df_features[["asin"] + CAT_LEVELS]

pair_cats = (
    co_pairs
    .assign(asinA=co_pairs["asinA"].astype(str), asinB=co_pairs["asinB"].astype(str))
    .merge(item_cats.add_prefix("a_"), left_on="asinA", right_on="a_asin", how="inner")
    .merge(item_cats.add_prefix("b_"), left_on="asinB", right_on="b_asin", how="inner")
    .drop(columns=["a_asin", "b_asin"])
)

print(f"co_purchase pairs            : {len(co_pairs):>12,}")
print(f"after joining features (both) : {len(pair_cats):>12,} "
      f"({len(pair_cats) / len(co_pairs):.1%})")
print(f"dropped, an asin has no features: {len(co_pairs) - len(pair_cats):>10,}")
print(f"\ncolumns ({pair_cats.shape[1]}): {list(pair_cats.columns)}")
pair_cats.head(5)

co_purchase pairs            :   10,595,885
after joining features (both) :    7,729,936 (73.0%)
dropped, an asin has no features:  2,865,949

columns (8): ['asinA', 'asinB', 'a_cat_2', 'a_cat_3', 'a_cat_4_clean', 'b_cat_2', 'b_cat_3', 'b_cat_4_clean']


,asinA,asinB,a_cat_2,a_cat_3,a_cat_4_clean,b_cat_2,b_cat_3,b_cat_4_clean
0,0560467893,B001E95R0O,Home Dcor,Home Dcor Accents,Corner Shelves,Furniture,Accent Furniture,Storage Trunks
1,0560467893,B001F7SGHQ,Home Dcor,Home Dcor Accents,Corner Shelves,Kitchen & Dining,Kitchen Utensils & Gadgets,Bar & Wine Tools
2,0560467893,B004RAL4H2,Home Dcor,Home Dcor Accents,Corner Shelves,Furniture,Bedroom Furniture,"Beds, Frames & Bases"
3,0560467893,B005C7SRLK,Home Dcor,Home Dcor Accents,Corner Shelves,Kitchen & Dining,Dining & Entertaining,Serveware
4,0560467893,B007EAROSK,Home Dcor,Home Dcor Accents,Corner Shelves,Bath,Bathroom Accessories,Holders & Dispensers


In [6]:
# The mapping's column names, from the package that built it — ROOT went on
# sys.path in the imports cell.
from complementary_cats_pairs import DST_COLS, SRC_COLS

# --- Steps 2 & 3: the two inner joins, then concatenate --------------------
A_CATS = [f"a_{c}" for c in CAT_LEVELS]
B_CATS = [f"b_{c}" for c in CAT_LEVELS]

QUERY_CATS = ["query_cat_2", "query_cat_3", "query_cat_4"]
TARGET_CATS = ["target_cat_2", "target_cat_3", "target_cat_4"]
FINAL_COLS = ["asin_query", "asin_target"] + QUERY_CATS + TARGET_CATS

# Only the six category columns are needed; the metrics are not model inputs.
mapping = comp_cat[SRC_COLS + DST_COLS].astype(str)


def directed_pairs(query_asin, query_cats, target_asin, target_cats):
    """Keep pairs whose (query path -> target path) is in the mapping.

    An inner join of `pair_cats` onto the mapping, with the named side lined up
    against `src_*` and the other against `dst_*`, then relabelled so the
    source side is always the query.
    """
    out = pair_cats.merge(
        mapping,
        left_on=query_cats + target_cats,
        right_on=SRC_COLS + DST_COLS,
        how="inner",
    )
    return out.rename(columns=dict(
        [(query_asin, "asin_query"), (target_asin, "asin_target")]
        + list(zip(query_cats, QUERY_CATS))
        + list(zip(target_cats, TARGET_CATS))
    ))[FINAL_COLS]


table_1 = directed_pairs("asinA", A_CATS, "asinB", B_CATS)   # asinA is source
table_2 = directed_pairs("asinB", B_CATS, "asinA", A_CATS)   # asinB is source

tower_pairs = pd.concat([table_1, table_2], ignore_index=True)
for c in QUERY_CATS + TARGET_CATS:
    tower_pairs[c] = tower_pairs[c].astype("category")

print(f"table 1 (asinA -> asinB) : {len(table_1):>12,}")
print(f"table 2 (asinB -> asinA) : {len(table_2):>12,}")
print(f"tower_pairs (concat)     : {len(tower_pairs):>12,}")
print(f"\ncolumns ({tower_pairs.shape[1]}): {list(tower_pairs.columns)}")
print(f"distinct query items   : {tower_pairs['asin_query'].nunique():,}")
print(f"distinct target items  : {tower_pairs['asin_target'].nunique():,}")
print(f"memory                 : "
      f"{tower_pairs.memory_usage(deep=True).sum() / 1e6:,.0f} MB")
tower_pairs.head(10)

table 1 (asinA -> asinB) :    1,458,143
table 2 (asinB -> asinA) :    1,458,492
tower_pairs (concat)     :    2,916,635

columns (8): ['asin_query', 'asin_target', 'query_cat_2', 'query_cat_3', 'query_cat_4', 'target_cat_2', 'target_cat_3', 'target_cat_4']
distinct query items   : 145,184
distinct target items  : 145,077
memory                 : 368 MB


,asin_query,asin_target,query_cat_2,query_cat_3,query_cat_4,target_cat_2,target_cat_3,target_cat_4
0,0560467893,B007EAROSK,Home Dcor,Home Dcor Accents,Corner Shelves,Bath,Bathroom Accessories,Holders & Dispensers
1,0560467893,B0150ZOXEI,Home Dcor,Home Dcor Accents,Corner Shelves,Bath,Bathroom Accessories,Holders & Dispensers
2,0681795107,B000Z4ETF8,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware
3,0681795107,B003ZYGQVK,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Kitchen & Dining,Cookware,Canning
4,0681795107,B00X5ETKU4,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Kitchen & Dining,"Coffee, Tea & Espresso",Coffee Makers
5,0768205921,B001JYMAA4,Home Dcor,Clocks,Missing,Kitchen & Dining,Storage & Organization,Food Storage
6,0768205921,B0054R3PVU,Home Dcor,Clocks,Missing,Home Dcor,Clocks,Alarm Clocks
7,0768205921,B00L0MIB8K,Home Dcor,Clocks,Missing,Home Dcor,Clocks,Wall Clocks
8,1574893122,B002HPNDCS,Wall Art,Posters & Prints,Missing,Wall Art,Posters & Prints,Missing
9,1574893122,B00BSF5S7G,Wall Art,Posters & Prints,Missing,Wall Art,Posters & Prints,Missing


## 5. Fold rare brands

`brand_clean` folds brands carried by ten or fewer items into `other_brands`,
taking 98,532 brands down to 12,747.

Nothing in this notebook consumes it yet — it is item-side preprocessing for
the product encoder, which reads `df_features` in both towers. Brand is a
natural encoder input alongside title, price and category, and an embedding
table needs the long tail bucketed: a brand on three items would otherwise get
a vector trained by a handful of gradient updates.

The pipeline also produces this now (`clean_brand` is Filter 5), so once you
re-extract, this cell recomputes what the pickle already carries.


In [ ]:
# Fold rare brands: keep those carried by MORE THAN 10 distinct items. A brand
# on three items would get an embedding trained by a handful of gradient
# updates; bucketing those into one `other_brands` symbol is more honest than
# pretending each has a learned vector.
MIN_ITEMS = 10
OTHER = "other_brands"

# Count on brand_norm where the pipeline produced it, so "3d rose" and "3drose"
# are not counted separately and pushed under the threshold by a split spelling.
source = "brand_norm" if "brand_norm" in df_features.columns else "brand"
brand_counts = df_features.groupby(source)["asin"].nunique()

kept = brand_counts[brand_counts > MIN_ITEMS].index
df_features["brand_clean"] = df_features[source].where(
    df_features[source].isin(kept) | df_features[source].isna(),
    OTHER,
)

n_before = df_features[source].nunique()
n_after = df_features["brand_clean"].nunique()
n_missing = df_features[source].isna().sum()
print(f"counted on : {source}")
print(f"brands     : {n_before:,} -> {n_after:,} "
      f"(kept {len(kept):,} with > {MIN_ITEMS} items, rest -> {OTHER!r})")
print(f"items      : {(df_features['brand_clean'] == OTHER).mean():.1%} in {OTHER}, "
      f"{n_missing / len(df_features):.1%} missing (left as NaN)")
print()
print(df_features["brand_clean"].value_counts().head(10))